## Publication and reproducibility notes

This notebook is part of the research source code for the **Comparative Disease Prediction** project.

- The original experiment logic and code-cell source are retained in this publication copy.
- Saved cell outputs have been cleared to remove machine-specific filesystem paths, verbose logs, and embedded data previews. This does **not** change the code or the experiment results recorded in the project documentation and reference-results files.
- The input datasets are not included in this repository. Obtain the permitted source files separately, review their licence/terms, and place them at the paths expected by the configuration cells.
- Run this notebook in a dedicated Python environment with the compatible dependencies installed. Do not assume the latest package versions are compatible with persisted model artefacts.
- Re-running the notebook performs the experiment (including CI testing and model fitting); it is not required to launch the separately packaged prediction application, which loads the saved trained models.
- This is a research prototype, not a clinical diagnostic system.

### Required input files

- `train_encoded.csv` (4,920 rows; 41 one-hot disease columns and 132 symptom columns in the audited candidate).

### Experiment summary

The notebook constructs a stratified 80/20 train/test split, runs its Stable-FCI procedure on the training split, extracts disease-adjacent symptoms from the final PAG, and trains Logistic Regression, Random Forest, and XGBoost classifiers. The configured maximum conditioning depth and maximum path length are both 10; these bounds must be reported when describing the method.

### Current input-schema behavior

The current implementation identifies disease columns by the `disease_` prefix and treats the remaining columns as symptom features. The `DISEASE_COLUMN` and `DISEASE_COLUMNS` entries in the configuration cell are legacy/documentary options and are not consumed by the current data-loading code. For this published experiment, provide the audited one-hot `disease_` columns; do not assume the categorical-label alternative described in the nearby comments is implemented.

### Data and results policy

Raw datasets, private files, environment folders, and machine-specific paths are intentionally not bundled. See the repository's data and reproducibility documentation for instructions on obtaining inputs and interpreting the archived results.


# Stable-FCI Causal Feature Selection → Multiclass Logistic Regression, Random Forest & XGBoost

This notebook keeps the Stable-FCI causal-discovery pipeline intact and uses **multiclass Logistic Regression**, **Random Forest**, and **XGBoost** as the downstream disease-prediction models, tuned and evaluated the same way on the same features and the same train/test split (Sections 12-21 for Logistic Regression, Sections 22-29 for Random Forest, Sections 31-38 for XGBoost, Section 39 for the three-way head-to-head comparison).

The two prediction branches are:

```text
TRAIN/TEST SPLIT
      │
      ├── TRAIN → Stable-FCI → final PAG → selected symptoms
      │                                      ↓
      │                              Logistic Regression
      │                              + L2 regularization
      │                              + 5-fold CV for C
      │                                      ↓
      │                                   TEST
      │
      └── TRAIN → all symptoms → Logistic Regression
                                   + L2 regularization
                                   + 5-fold CV for C
                                           ↓
                                         TEST

                  Compare TEST performance
```

The test set is kept untouched until the final evaluation.

In [ ]:
# ---------------------------------------------------------------------------
# Install dependencies (safe to re-run)
# ---------------------------------------------------------------------------
# IMPORTANT: this pipeline requires causal-learn >= 0.1.4.0 -- that is the version
# that introduced ruleR5, ruleR6, ruleR7, rule8, rule9 and rule10 in
# causallearn.search.ConstraintBased.FCI (older versions only have rule0..ruleR4B and
# will raise "ImportError: cannot import name 'ruleR5'" on the Imports cell below).
# A stale, older causal-learn can already be present in a Colab runtime, and plain
# `pip install causal-learn` silently skips upgrading an already-installed package --
# so this cell explicitly pins and force-checks the version instead.
import sys, subprocess

REQUIRED_CAUSAL_LEARN = "0.1.4.8"  # latest verified-working version as of this notebook

def _pip_install(spec):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", spec],
                    check=True)

for _pkg in [f"causal-learn=={REQUIRED_CAUSAL_LEARN}", "joblib", "xgboost"]:
    try:
        _pip_install(_pkg)
    except Exception as _e:
        print(f"Warning: could not install {_pkg}: {_e}")

# Verify the *active* installed version actually satisfies the requirement -- catches
# the case where causal-learn was already imported earlier in this Colab session (in
# which case Python will keep using the old in-memory module until the runtime is
# restarted, even though pip just upgraded the on-disk package).
import importlib.metadata as _im
_installed = _im.version("causal-learn")
print(f"causal-learn installed version: {_installed}")
if tuple(map(int, _installed.split("."))) < tuple(map(int, REQUIRED_CAUSAL_LEARN.split("."))):
    print("WARNING: installed causal-learn is older than required. If you still see "
          "this after re-running this cell, go to Runtime > Restart runtime in Colab "
          "and run all cells again from the top.")
else:
    print("causal-learn version OK.")


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import json
import time
import contextlib
import itertools
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Circle as MplCircle
from matplotlib.collections import LineCollection

from joblib import Parallel, delayed

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    top_k_accuracy_score
)
import joblib
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from causallearn.graph.GraphClass import CausalGraph
from causallearn.graph.Endpoint import Endpoint
from causallearn.utils.cit import CIT
from causallearn.utils.PCUtils.Helper import append_value
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.search.ConstraintBased.FCI import (
    rule0, removeByPossibleDsep,
    rulesR1R2cycle, ruleR3, ruleR4B, ruleR5, ruleR6, ruleR7,
    rule8, rule9, rule10
)

warnings.filterwarnings("ignore", category=RuntimeWarning)
print("Imports OK.")

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION -- edit this cell for your environment / run
# ---------------------------------------------------------------------------
CONFIG = {
    # ---- data ----
    "DATA_PATH": "train_encoded.csv",  # <-- EDIT ME
    # If your dataset already stores diseases as one-hot columns, list them here and
    # leave DISEASE_COLUMN as None. If instead you have a single categorical column
    # (e.g. "prognosis") naming the disease, set DISEASE_COLUMN to that column's name
    # and leave DISEASE_COLUMNS as None -- it will be one-hot encoded automatically.
    "DISEASE_COLUMN": "prognosis",
    "DISEASE_COLUMNS": None,
    "DISEASE_PREFIX": "disease_",

    # ---- train/test split ----
    "TEST_SIZE": 0.20,
    "RANDOM_STATE": 42,

    # ---- Stable-FCI ----
    "CI_TEST_METHOD": "gsq",     # G^2 (likelihood-ratio) test, appropriate for binary data
    "ALPHA": 0.05,               # significance level for independence
    "MAX_DEPTH": 10,               # -1 = unlimited (true Stable-FCI); capped for tractability
                                   # on ~173 binary variables in a notebook/Colab session.
                                   # Raise this (or set to -1) if you have the compute budget.
    "MAX_PATH_LENGTH": 10,        # for discriminating-path search (rule R4), -1 = unlimited
    "N_JOBS": -1
    ,                 # parallel workers for CI tests within each depth (-1 = all cores)

    # ---- prediction model: multiclass Logistic Regression ----
    # Smaller C = stronger L2 regularization; larger C = weaker regularization.
    "LR_C_GRID": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    "LR_CV_FOLDS": 5,
    "LR_MAX_ITER": 5000,
    "LR_SOLVER": "lbfgs",
    "LR_PENALTY": "l2",
    "LR_SCORING": "macro_f1",

    # ---- prediction model: Random Forest ----
    # Tuned by 5-fold stratified CV over (n_estimators, max_depth) pairs,
    # the same way LR is tuned over its C grid.
    "RF_PARAM_GRID": [
        {"n_estimators": 100, "max_depth": None},
        {"n_estimators": 100, "max_depth": 10},
        {"n_estimators": 300, "max_depth": None},
        {"n_estimators": 300, "max_depth": 10},
        {"n_estimators": 300, "max_depth": 20},
        {"n_estimators": 500, "max_depth": None},
    ],
    "RF_CV_FOLDS": 5,
    "RF_MIN_SAMPLES_LEAF": 1,
    "RF_N_JOBS": -1,
    "RF_SCORING": "macro_f1",

    # ---- prediction model: XGBoost ----
    # Tuned by 5-fold stratified CV over (n_estimators, max_depth, learning_rate)
    # triples, the same way RF is tuned over its param grid.
    "XGB_PARAM_GRID": [
        {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.3},
        {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.1},
        {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.1},
        {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05},
        {"n_estimators": 500, "max_depth": 6, "learning_rate": 0.05},
        {"n_estimators": 500, "max_depth": 9, "learning_rate": 0.01},
    ],
    "XGB_CV_FOLDS": 5,
    "XGB_SUBSAMPLE": 0.8,
    "XGB_COLSAMPLE_BYTREE": 0.8,
    "XGB_N_JOBS": -1,
    "XGB_SCORING": "macro_f1",

    # ---- results ----
    "RESULTS_ROOT": "results",

    # ---- Colab / Google Drive ----
    "USE_GOOGLE_DRIVE": True,
}

print(json.dumps(CONFIG, indent=2))


In [ ]:
# ---------------------------------------------------------------------------
# Mount Google Drive (Colab only -- safely skipped elsewhere)
# ---------------------------------------------------------------------------
if CONFIG["USE_GOOGLE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print("Not running in Colab (or google.colab unavailable) -- skipping Drive mount.")


In [ ]:
# ---------------------------------------------------------------------------
# Create timestamped results directory with the requested folder structure
# ---------------------------------------------------------------------------
RUN_ID = datetime.now().strftime("run_%Y_%m_%d_%H_%M_%S")
RUN_DIR = Path(CONFIG["RESULTS_ROOT"]) / RUN_ID
LR_DIR = RUN_DIR / "logistic_regression"
RF_DIR = RUN_DIR / "random_forest"
XGB_DIR = RUN_DIR / "xgboost"
RUN_DIR.mkdir(parents=True, exist_ok=True)
LR_DIR.mkdir(parents=True, exist_ok=True)
RF_DIR.mkdir(parents=True, exist_ok=True)
XGB_DIR.mkdir(parents=True, exist_ok=True)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

RUNTIME = {}  # stage_name -> seconds, filled in throughout the notebook

def mark_time(stage, t0):
    RUNTIME[stage] = round(time.time() - t0, 4)
    print(f"[{stage}] took {RUNTIME[stage]:.2f}s")

print(f"Results will be written to: {RUN_DIR.resolve()}")


## 1. Data Loading & Validation

Loads the raw dataset, checks for missing values, verifies binary encoding, identifies
disease vs. symptom columns, removes irrelevant columns, and writes `data_summary.json`.


In [ ]:
# ---------------------------------------------------------------------------
# Load dataset
# ---------------------------------------------------------------------------
t0 = time.time()

raw_df = pd.read_csv(CONFIG["DATA_PATH"])
n_rows_original, n_cols_original = raw_df.shape

disease_cols = [c for c in raw_df.columns if c.startswith("disease_")]
symptom_cols = [c for c in raw_df.columns if c not in disease_cols]

full_df = raw_df[disease_cols + symptom_cols].astype(int)

data_summary = {
    "n_rows_original": int(n_rows_original),
    "n_cols_original": int(n_cols_original),
    "n_disease_columns": len(disease_cols),
    "n_symptom_columns": len(symptom_cols),
    "disease_columns": disease_cols,
    "symptom_columns": symptom_cols,
    "n_rows_final": int(len(full_df)),
}
save_json(data_summary, RUN_DIR / "data_summary.json")
mark_time("data_validation", t0)
print(f"Final modeling dataframe shape: {full_df.shape}")
full_df.head()

In [ ]:
# ---------------------------------------------------------------------------
# Reconstruct a single multiclass disease label (used for stratification & classification)
# and a node_type lookup (disease/symptom) used throughout the rest of the notebook.
# ---------------------------------------------------------------------------
node_type = {c: "disease" for c in disease_cols}
node_type.update({c: "symptom" for c in symptom_cols})

disease_label = full_df[disease_cols].idxmax(axis=1).str.replace(
    CONFIG["DISEASE_PREFIX"], "", regex=False)
full_df["_disease_label"] = disease_label

print(f"Number of distinct diseases: {disease_label.nunique()}")
print(disease_label.value_counts().head(10))


## 2. Train / Test Split

80% train / 20% test, **stratified by disease**. The test set is kept completely aside --
Stable-FCI is run on the training data only; the test set is never touched until final
model evaluation.


In [ ]:
# ---------------------------------------------------------------------------
# Stratified 80/20 train/test split
# ---------------------------------------------------------------------------
t0 = time.time()

all_feature_cols = disease_cols + symptom_cols
y_disease = full_df["_disease_label"]

train_idx, test_idx = train_test_split(
    full_df.index,
    test_size=CONFIG["TEST_SIZE"],
    random_state=CONFIG["RANDOM_STATE"],
    stratify=y_disease,
)

train_df = full_df.loc[train_idx].reset_index(drop=True)
test_df = full_df.loc[test_idx].reset_index(drop=True)

train_test_split_summary = {
    "test_size": CONFIG["TEST_SIZE"],
    "random_state": CONFIG["RANDOM_STATE"],
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
    "train_class_counts": train_df["_disease_label"].value_counts().to_dict(),
    "test_class_counts": test_df["_disease_label"].value_counts().to_dict(),
}
save_json(train_test_split_summary, RUN_DIR / "train_test_split.json")
mark_time("train_test_split", t0)
print(f"Train: {train_df.shape}, Test: {test_df.shape}")


## 3. Graph Layout & Drawing Utilities

A single fixed layout (diseases on an inner ring, symptoms on an outer ring) is computed
once and reused for every diagram (`complete_graph.pdf`, `stable_fci_skeleton.pdf`,
`possible_dsep_graph.pdf`, `final_pag.pdf`) so node positions are directly comparable
across stages. Edge marks follow standard PAG notation: **tail** (no mark), **arrow**
(arrowhead), **circle** (open circle, unknown mark).


In [ ]:
# ---------------------------------------------------------------------------
# Layout + PAG drawing helpers
# ---------------------------------------------------------------------------
def compute_layout(disease_cols, symptom_cols):
    pos = {}
    n_d, n_s = len(disease_cols), len(symptom_cols)
    for i, name in enumerate(sorted(disease_cols)):
        angle = 2 * np.pi * i / max(n_d, 1)
        pos[name] = (1.0 * np.cos(angle), 1.0 * np.sin(angle))
    for i, name in enumerate(sorted(symptom_cols)):
        angle = 2 * np.pi * i / max(n_s, 1)
        pos[name] = (2.6 * np.cos(angle), 2.6 * np.sin(angle))
    return pos

LAYOUT = compute_layout(disease_cols, symptom_cols)

def _draw_mark(ax, node_x, node_y, mark_x, mark_y, ep):
    if ep == "ARROW":
        arrow = FancyArrowPatch((mark_x, mark_y), (node_x, node_y), arrowstyle="-|>",
                                 mutation_scale=9, color="black", zorder=2, linewidth=0.6,
                                 shrinkA=0, shrinkB=0)
        ax.add_patch(arrow)
    elif ep == "CIRCLE":
        circ = MplCircle((mark_x, mark_y), radius=0.025, fill=False, color="black",
                          linewidth=0.8, zorder=2)
        ax.add_patch(circ)
    # TAIL -> no mark drawn

def draw_pag(edges_list, pos, node_type, title, filepath, figsize=(16, 16),
             node_size=160, show_symptom_labels=True):
    fig, ax = plt.subplots(figsize=figsize)
    segments = [[pos[n1], pos[n2]] for (n1, n2, ep1, ep2) in edges_list if n1 in pos and n2 in pos]
    if segments:
        lc = LineCollection(segments, colors="#999999", linewidths=0.5, alpha=0.55, zorder=1)
        ax.add_collection(lc)
    for (n1, n2, ep1, ep2) in edges_list:
        if n1 not in pos or n2 not in pos:
            continue
        x1, y1 = pos[n1]
        x2, y2 = pos[n2]
        dx, dy = x2 - x1, y2 - y1
        dist = (dx ** 2 + dy ** 2) ** 0.5
        if dist == 0:
            continue
        ux, uy = dx / dist, dy / dist
        if ep1 != "TAIL":
            _draw_mark(ax, x1, y1, x1 + ux * 0.09, y1 + uy * 0.09, ep1)
        if ep2 != "TAIL":
            _draw_mark(ax, x2, y2, x2 - ux * 0.09, y2 - uy * 0.09, ep2)

    names = list(pos.keys())
    xs = [pos[n][0] for n in names]
    ys = [pos[n][1] for n in names]
    colors = ["#d1495b" if node_type.get(n) == "disease" else "#1f77b4" for n in names]
    ax.scatter(xs, ys, s=node_size, c=colors, zorder=3, edgecolors="black", linewidths=0.4)

    for n in names:
        if node_type.get(n) == "disease" or show_symptom_labels:
            x, y = pos[n]
            label = n.replace(CONFIG["DISEASE_PREFIX"], "")
            ax.annotate(label, (x, y), fontsize=4.5, ha="center", va="center", zorder=4)

    ax.legend(handles=[mpatches.Patch(color="#d1495b", label="Disease"),
                        mpatches.Patch(color="#1f77b4", label="Symptom")],
              loc="upper right", fontsize=9)
    ax.set_title(title, fontsize=14)
    ax.set_xlim(min(xs) - 0.6, max(xs) + 0.6)
    ax.set_ylim(min(ys) - 0.6, max(ys) + 0.6)
    ax.set_aspect("equal")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(filepath, format="pdf")
    plt.close(fig)
    print(f"Saved {filepath}")

def get_edges_info(graph):
    out = []
    for e in graph.get_graph_edges():
        out.append((e.get_node1().get_name(), e.get_node2().get_name(),
                    e.get_endpoint1().name, e.get_endpoint2().name))
    return out


## 4. Complete Graph (Stable-FCI Input)

Before any CI testing, Stable-FCI starts from the **fully connected graph** over all
disease + symptom nodes (edges undirected / tail-tail). This is the input state to
Stable-FAS.


In [ ]:
# ---------------------------------------------------------------------------
# Draw & save the initial complete graph
# ---------------------------------------------------------------------------
t0 = time.time()
all_nodes = disease_cols + symptom_cols
complete_edges = [(a, b, "TAIL", "TAIL") for a, b in itertools.combinations(sorted(all_nodes), 2)]
print(f"Complete graph: {len(all_nodes)} nodes, {len(complete_edges)} edges")

draw_pag(complete_edges, LAYOUT, node_type,
         f"Complete Graph (Stable-FCI Input) -- {len(all_nodes)} nodes, "
         f"{len(complete_edges)} edges",
         RUN_DIR / "complete_graph.pdf", show_symptom_labels=False)
mark_time("complete_graph_plot", t0)


## 5. Stable-FAS with Depth-Wise Parallel CI Testing

This is the core algorithmic contribution of this notebook: a **stable** adjacency search
(matching `causal-learn`'s own `stable=True` semantics exactly -- verified against the
reference implementation) in which, at each depth `d`:

1. The neighbor sets of every node are **frozen** (snapshotted) before any testing begins.
2. The full list of `(X, Y, S)` CI-test tasks for depth `d` is built from that frozen
   snapshot.
3. Tasks are grouped per ordered pair `(X, Y)` and dispatched to a `joblib.Parallel` pool
   (process-based `loky` backend -- safe for Jupyter/Colab, bypasses the GIL for true
   parallel speed-up). Each worker tests every conditioning set `S` of size `d` drawn from
   `adj(X) \ {Y}` and reports, for its pair, every `(S, p-value)` plus the union of all
   `S` for which `X` and `Y` tested independent (`p > alpha`).
4. Only **after every task for depth `d` has returned** are edges removed from the graph,
   exactly mirroring the sequential `stable=True` algorithm -- so parallelizing does not
   change the result, only the wall-clock time.

At depth 0, `(X, Y)` and `(Y, X)` are equivalent (both test the empty conditioning set), so
they are de-duplicated; at depth >= 1 both directions are tested separately since `adj(X)`
and `adj(Y)` generally differ (this matches the reference `causal-learn` implementation and
is required for correctness/completeness of the stable search).

The **Possible-D-SEP** step later in the FCI pipeline is, by contrast, inherently
sequential (removing an edge changes the Possible-D-SEP sets of other edges), so it is run
sequentially using the reference implementation further down, as it should be.


In [ ]:
# ---------------------------------------------------------------------------
# Worker-side CIT cache: avoids reconstructing the CIT object on every task
# dispatched to the same persistent worker process.
# ---------------------------------------------------------------------------
_CIT_CACHE = {}

def _get_cit(data, method):
    key = (id(data), method)
    if key not in _CIT_CACHE:
        _CIT_CACHE[key] = CIT(data, method=method)
    return _CIT_CACHE[key]


def _test_pair(x, y, neigh_x, current_depth, data, method, alpha):
    # Runs every CI test needed for ordered pair (x, y) at this depth, against the
    # FROZEN neighbor set neigh_x. Returns everything the driver needs to update sep_sets
    # and the edge-removal set -- but does NOT mutate any shared graph state, which is
    # exactly what makes this call safe to parallelize.
    cit = _get_cit(data, method)
    neigh_x_noy = [n for n in neigh_x if n != y]
    tested = []
    sepset_elems = set()
    found_independent = False
    for S in itertools.combinations(neigh_x_noy, current_depth):
        p = cit(x, y, S)
        tested.append((S, p))
        if p > alpha:
            found_independent = True
            sepset_elems.update(S)
    return x, y, tested, tuple(sepset_elems), found_independent


def parallel_stable_fas(data, node_names, method="gsq", alpha=0.05, depth=-1, n_jobs=-1,
                         verbose=True):
    # Order-independent ('stable') Fast Adjacency Search with depth-wise parallel CI
    # testing. Reproduces causal-learn's fas(..., stable=True) result exactly (validated
    # against the reference implementation), but parallelizes the independent CI tests
    # within each depth using joblib.
    #
    # Returns
    # -------
    # cg : CausalGraph  (cg.G is the skeleton graph, all edges TAIL-TAIL at this point)
    # sep_sets : dict[(int,int) -> set[int]]
    # ci_log : list[dict]  (flat log of every CI test performed, for ci_tests.csv)
    no_of_var = data.shape[1]
    cg = CausalGraph(no_of_var, node_names)
    cg.set_ind_test(CIT(data, method=method))
    sep_sets = {}
    ci_log = []

    max_depth = float("inf") if depth == -1 else depth
    current_depth = -1

    while cg.max_degree() - 1 > current_depth and current_depth < max_depth:
        current_depth += 1

        # --- 1. FREEZE the adjacency structure for this depth ---
        frozen_neigh = {x: cg.neighbors(x).tolist() for x in range(no_of_var)}

        # --- 2. Build the (independent) task list for this depth ---
        tasks = []
        seen_unordered = set()
        for x in range(no_of_var):
            Neigh_x = frozen_neigh[x]
            if len(Neigh_x) - 1 < current_depth:
                continue
            for y in Neigh_x:
                if current_depth == 0:
                    key = tuple(sorted((x, y)))
                    if key in seen_unordered:
                        continue
                    seen_unordered.add(key)
                tasks.append((x, y))

        if verbose:
            print(f"Depth {current_depth}: {len(tasks)} independent (X,Y) CI-test tasks "
                  f"to dispatch across {n_jobs if n_jobs != -1 else 'all'} workers...")

        if not tasks:
            continue

        # --- 3. Dispatch all of this depth's tasks in parallel ---
        results = Parallel(n_jobs=n_jobs)(
            delayed(_test_pair)(x, y, frozen_neigh[x], current_depth, data, method, alpha)
            for (x, y) in tasks
        )

        # --- 4. Collect results; edges are NOT removed until every task has returned ---
        edge_removal = set()
        for x, y, tested, sepset_elems, found_independent in results:
            for (S, p) in tested:
                ci_log.append(dict(
                    depth=current_depth, X=node_names[x], Y=node_names[y],
                    S=";".join(node_names[i] for i in S), p_value=float(p),
                    independent=bool(p > alpha),
                ))
            if found_independent:
                edge_removal.add((x, y))
                edge_removal.add((y, x))
            if sepset_elems or found_independent:
                append_value(cg.sepset, x, y, tuple(sepset_elems))
                append_value(cg.sepset, y, x, tuple(sepset_elems))

        # --- 5. NOW remove edges (after the whole depth is done) ---
        n_removed = 0
        for (x, y) in edge_removal:
            edge = cg.G.get_edge(cg.G.nodes[x], cg.G.nodes[y])
            if edge is not None:
                cg.G.remove_edge(edge)
                n_removed += 1
            if cg.sepset[x, y] is not None:
                origin_set = set(l_in for l_out in cg.sepset[x, y] for l_in in l_out)
                sep_sets[(x, y)] = origin_set
                sep_sets[(y, x)] = origin_set

        if verbose:
            print(f"  -> removed {n_removed // 2} edges at depth {current_depth} "
                  f"(remaining max degree: {int(cg.max_degree())})")

    return cg, sep_sets, ci_log

print("parallel_stable_fas defined.")


In [ ]:
# ---------------------------------------------------------------------------
# Run parallel Stable-FAS on TRAINING DATA ONLY (all disease + symptom nodes)
# ---------------------------------------------------------------------------
t0 = time.time()

fci_node_names = disease_cols + symptom_cols
train_data_matrix = train_df[fci_node_names].to_numpy(dtype=np.float64)

print(f"Running parallel Stable-FAS on training data: {train_data_matrix.shape[0]} patients, "
      f"{train_data_matrix.shape[1]} variables ({len(disease_cols)} diseases + "
      f"{len(symptom_cols)} symptoms)")
print(f"CI test: {CONFIG['CI_TEST_METHOD']}, alpha={CONFIG['ALPHA']}, "
      f"max_depth={CONFIG['MAX_DEPTH']}, n_jobs={CONFIG['N_JOBS']}")

cg, sep_sets, ci_log = parallel_stable_fas(
    train_data_matrix, fci_node_names,
    method=CONFIG["CI_TEST_METHOD"], alpha=CONFIG["ALPHA"],
    depth=CONFIG["MAX_DEPTH"], n_jobs=CONFIG["N_JOBS"], verbose=True,
)

skeleton_graph = cg.G
graph_nodes = cg.G.nodes  # index-aligned with fci_node_names / sep_sets integer indices

mark_time("stable_fas_parallel", t0)
print(f"\nStable skeleton: {len(skeleton_graph.get_graph_edges())} edges remaining "
      f"(from {len(fci_node_names) * (len(fci_node_names) - 1) // 2} possible), "
      f"{len(ci_log)} total CI tests performed.")


In [ ]:
# ---------------------------------------------------------------------------
# Save ci_tests.csv, sepsets.json, and the stable skeleton diagram
# ---------------------------------------------------------------------------
ci_tests_df = pd.DataFrame(ci_log)
ci_tests_df.to_csv(RUN_DIR / "ci_tests.csv", index=False)
print(f"Saved ci_tests.csv ({len(ci_tests_df)} rows)")

sepsets_out = {}
for (xi, yi), s in sep_sets.items():
    if xi < yi:
        key = f"{fci_node_names[xi]}|{fci_node_names[yi]}"
        sepsets_out[key] = sorted(fci_node_names[i] for i in s)
save_json(sepsets_out, RUN_DIR / "sepsets.json")
print(f"Saved sepsets.json ({len(sepsets_out)} separated pairs)")

skeleton_edges_info = get_edges_info(skeleton_graph)
draw_pag(skeleton_edges_info, LAYOUT, node_type,
         f"Stable-FCI Skeleton (after Stable-FAS) -- {len(skeleton_edges_info)} edges",
         RUN_DIR / "stable_fci_skeleton.pdf")


## 6. Domain Knowledge as FCI Background Knowledge

Medical domain knowledge says a **disease causes its symptoms**, not the reverse. Rather
than letting the data-driven orientation rules run to completion and then overwriting
their conclusion, the constraint is registered as **tiered background knowledge** with
`causal-learn`'s own `BackgroundKnowledge` class *before* orientation starts:

- All **disease** nodes go in tier 0 ("can be a cause").
- All **symptom** nodes go in tier 1 ("can be an effect").

`BackgroundKnowledge.is_forbidden(X, Y)` then automatically returns `True` whenever `X` is
a later tier than `Y` -- i.e. `Symptom -> Disease` is forbidden, `Disease -> Symptom`
is not. Every orientation function below (`rule0`, `rulesR1R2cycle`, `ruleR3`, `ruleR4B`)
is FCI-background-knowledge-aware: `rule0` forces the correct direction the moment an
edge is first considered (before collider detection even happens for that edge), and every
later rule consults `is_arrow_point_allowed(...)` so it can never re-introduce the
forbidden direction while propagating further orientations. This produces a PAG that is
genuinely closed under the FCI rules *given* the constraint, rather than one where the
constraint was bolted on after the fact.


In [ ]:
# ---------------------------------------------------------------------------
# Build tiered background knowledge: Disease (tier 0) causally precedes Symptom (tier 1)
# ---------------------------------------------------------------------------
background_knowledge = BackgroundKnowledge()
for node in graph_nodes:
    tier = 0 if node_type[node.get_name()] == "disease" else 1
    background_knowledge.add_node_to_tier(node, tier)

print(f"Background knowledge: {len(disease_cols)} disease nodes in tier 0, "
      f"{len(symptom_cols)} symptom nodes in tier 1 "
      f"(Symptom -> Disease is now forbidden for every orientation rule below).")


## 7. Collider Detection & Possible-D-SEP

Starting from the stable skeleton:

1. **Rule 0** reorients every edge to `o-o`, immediately force-orients any disease-symptom
   edge per the background knowledge above, then marks unshielded colliders
   (`X *-> Y <-* Z` where X, Z not adjacent and Y not in `sepset(X,Z)`) using the
   separating sets found during Stable-FAS.
2. **Possible-D-SEP** is computed for each remaining edge and additional G^2 tests are run
   against conditioning sets drawn from it, which can remove edges the adjacency search
   missed (this is what distinguishes FCI from plain PC and lets it handle latent
   confounders correctly).
3. **Rule 0** is re-applied on the (possibly sparser) graph.

Possible-D-SEP itself does not need the background knowledge (it only removes edges via
CI tests, it does not orient anything), and is intentionally **not parallelized**: removing
an edge changes the Possible-D-SEP sets of other edges, so the tests are sequentially
dependent. Parallelizing it would either be incorrect (stale Possible-D-SEP sets) or
require re-synchronizing after every single edge removal, which offers no real speed-up --
the parallelism this pipeline needed belongs to the depth-wise Stable-FAS step above, where
it is both safe and effective.


In [ ]:
# ---------------------------------------------------------------------------
# Collider detection (rule0, knowledge-aware) + Possible-D-SEP + additional G^2 tests
# ---------------------------------------------------------------------------
from causallearn.search.ConstraintBased.FCI import getPossibleDsep
from causallearn.utils.DepthChoiceGenerator import DepthChoiceGenerator

def _contains_all(set_a, set_b):
    return set_b.issubset(set_a)

def removeByPossibleDsep_bounded(graph, independence_test_method, alpha, sep_sets,
                                  max_path_length=-1, max_cond_set_size=-1):
    n_removed = 0
    for edge in graph.get_graph_edges():
        node_a, node_b = edge.get_node1(), edge.get_node2()
        for x_node, y_node in ((node_a, node_b), (node_b, node_a)):
            if not graph.contains_edge(edge):
                break
            possibleDsep = getPossibleDsep(x_node, y_node, graph, max_path_length)
            cap = len(possibleDsep) if max_cond_set_size == -1 else min(max_cond_set_size, len(possibleDsep))
            gen = DepthChoiceGenerator(len(possibleDsep), cap)
            choice = gen.next()
            while choice is not None:
                origin_choice = choice
                choice = gen.next()
                if len(origin_choice) < 2:
                    continue
                sepset = tuple(possibleDsep[i] for i in origin_choice)
                if _contains_all(set(graph.get_adjacent_nodes(node_a)), set(sepset)):
                    continue
                if _contains_all(set(graph.get_adjacent_nodes(node_b)), set(sepset)):
                    continue
                X, Y = graph.get_node_map()[node_a], graph.get_node_map()[node_b]
                condSet_index = tuple(graph.get_node_map()[possibleDsep[i]] for i in origin_choice)
                p_value = independence_test_method(X, Y, condSet_index)
                if p_value > alpha:
                    graph.remove_edge(edge)
                    sep_sets[(X, Y)] = set(condSet_index)
                    n_removed += 1
                    break
    return n_removed


t0 = time.time()

pag_graph = skeleton_graph

_bk_log = io.StringIO()
with contextlib.redirect_stdout(_bk_log):
    rule0(pag_graph, graph_nodes, sep_sets, background_knowledge, False)

indep_test = CIT(train_data_matrix, method=CONFIG["CI_TEST_METHOD"])
n_removed = removeByPossibleDsep_bounded(
    pag_graph, indep_test, CONFIG["ALPHA"], sep_sets,
    max_path_length=CONFIG["MAX_PATH_LENGTH"], max_cond_set_size=CONFIG["MAX_DEPTH"])
print(f"Possible-D-SEP removed {n_removed} additional edges "
      f"(max_path_length={CONFIG['MAX_PATH_LENGTH']}, max_cond_set_size={CONFIG['MAX_DEPTH']}).")

with contextlib.redirect_stdout(_bk_log):
    rule0(pag_graph, graph_nodes, sep_sets, background_knowledge, False)

n_bk_oriented = _bk_log.getvalue().count("Orienting edge (Knowledge)")
mark_time("possible_dsep_and_colliders", t0)
print(f"Background knowledge force-oriented {n_bk_oriented} disease-symptom edge "
      f"applications across both rule0 passes.")
print(f"Graph after Possible-D-SEP + collider detection: "
      f"{len(pag_graph.get_graph_edges())} edges")

possible_dsep_edges_info = get_edges_info(pag_graph)
draw_pag(possible_dsep_edges_info, LAYOUT, node_type,
         f"After Possible-D-SEP & Collider Detection -- {len(possible_dsep_edges_info)} edges",
         RUN_DIR / "possible_dsep_graph.pdf")

## 8. Orientation Rules (R1-R10) -> Final PAG

Applies the full FCI orientation rule set (as in `causal-learn`'s reference `fci()`
driver, now **knowledge-aware** throughout) until no rule produces further changes,
yielding the **Partial Ancestral Graph (PAG)**: edges with definite tails/arrows where the
rules (and the disease-precedes-symptom background knowledge) could determine them, and
`o-o` / `o->` marks where the direction remains genuinely undetermined given the data (and
the possibility of latent confounders, which FCI is designed to tolerate).


In [ ]:
# ---------------------------------------------------------------------------
# Apply orientation rules R1-R10 (knowledge-aware) until fixpoint
# ---------------------------------------------------------------------------
t0 = time.time()

change_flag = True
first_time = True
n_iter = 0
while change_flag:
    n_iter += 1
    change_flag = False
    change_flag = rulesR1R2cycle(pag_graph, background_knowledge, change_flag, False)
    change_flag = ruleR3(pag_graph, sep_sets, background_knowledge, change_flag, False)

    if change_flag or first_time:
        change_flag = ruleR4B(pag_graph, CONFIG["MAX_PATH_LENGTH"], train_data_matrix,
                               indep_test, CONFIG["ALPHA"], sep_sets, change_flag,
                               background_knowledge, False)
        first_time = False

    change_flag = ruleR5(pag_graph, change_flag, False)
    change_flag = ruleR6(pag_graph, change_flag, False)
    change_flag = ruleR7(pag_graph, change_flag, False)
    change_flag = rule8(pag_graph, graph_nodes, change_flag)
    change_flag = rule9(pag_graph, graph_nodes, change_flag)
    change_flag = rule10(pag_graph, change_flag)

    if n_iter > 50:
        print("WARNING: orientation rules did not converge within 50 iterations; stopping.")
        break

pag_graph.set_pag(True)
mark_time("orientation_rules", t0)
print(f"Orientation converged after {n_iter} iterations. "
      f"{len(pag_graph.get_graph_edges())} edges in the final PAG.")


## 9. Sanity Check: Domain Knowledge Was Actually Respected

Because the disease-precedes-symptom constraint was enforced *inside* the algorithm (Section
6-7) rather than patched on afterward, every disease-symptom edge in the final PAG should
already be oriented `Disease -> Symptom` (or still `o->`/`o-o` if the data never determined
a definite direction and the constraint alone didn't force one -- background knowledge
forbids the *wrong* direction, it does not fabricate certainty the rules didn't earn).
The one thing that should **never** happen is a definite `Symptom -> Disease` edge. This
cell verifies that directly against the final graph, rather than assuming it.


In [ ]:
# ---------------------------------------------------------------------------
# Verify: no Symptom -> Disease directed edge exists anywhere in the final PAG
# ---------------------------------------------------------------------------
final_edges_info = get_edges_info(pag_graph)

violations = []
for (n1, n2, ep1, ep2) in final_edges_info:
    t1, t2 = node_type.get(n1), node_type.get(n2)
    if {t1, t2} != {"disease", "symptom"}:
        continue
    if t1 == "symptom" and ep1 == "TAIL" and ep2 == "ARROW":
        violations.append((n1, n2))
    if t2 == "symptom" and ep2 == "TAIL" and ep1 == "ARROW":
        violations.append((n2, n1))

if violations:
    raise RuntimeError(
        f"Domain-knowledge violation: found {len(violations)} Symptom -> Disease "
        f"directed edge(s) despite background knowledge: {violations[:5]} ..."
    )
print(f"Verified: 0 Symptom -> Disease violations among "
      f"{sum(1 for n1,n2,e1,e2 in final_edges_info if {node_type.get(n1), node_type.get(n2)} == {'disease','symptom'})} "
      f"disease-symptom edges in the final PAG.")

draw_pag(final_edges_info, LAYOUT, node_type,
         f"Final PAG (Stable-FCI, background-knowledge-constrained) -- "
         f"{len(final_edges_info)} edges",
         RUN_DIR / "final_pag.pdf")


## 10. Edge Extraction

Classifies every edge in the final PAG by its endpoint marks and writes them out as CSVs.


In [ ]:
# ---------------------------------------------------------------------------
# all_edges.csv / directed_edges.csv / bidirected_edges.csv / uncertain_edges.csv
# ---------------------------------------------------------------------------
all_edges_rows = []
directed_rows = []
bidirected_rows = []
uncertain_rows = []

for (n1, n2, ep1, ep2) in final_edges_info:
    row = {"node1": n1, "node2": n2, "endpoint1": ep1, "endpoint2": ep2,
           "node1_type": node_type.get(n1), "node2_type": node_type.get(n2)}
    all_edges_rows.append(row)

    if (ep1 == "TAIL" and ep2 == "ARROW") or (ep1 == "ARROW" and ep2 == "TAIL"):
        src, dst = (n1, n2) if ep2 == "ARROW" else (n2, n1)
        directed_rows.append({**row, "source": src, "target": dst})
    elif ep1 == "ARROW" and ep2 == "ARROW":
        bidirected_rows.append(row)
    else:
        # at least one CIRCLE endpoint -> orientation genuinely undetermined
        uncertain_rows.append(row)

pd.DataFrame(all_edges_rows).to_csv(RUN_DIR / "all_edges.csv", index=False)
pd.DataFrame(directed_rows).to_csv(RUN_DIR / "directed_edges.csv", index=False)
pd.DataFrame(bidirected_rows).to_csv(RUN_DIR / "bidirected_edges.csv", index=False)
pd.DataFrame(uncertain_rows).to_csv(RUN_DIR / "uncertain_edges.csv", index=False)

print(f"all_edges: {len(all_edges_rows)} | directed: {len(directed_rows)} | "
      f"bidirected: {len(bidirected_rows)} | uncertain: {len(uncertain_rows)}")


## 11. Extract Stable-FCI-Selected Symptoms

**Selection rule:** a symptom is selected if it has **any remaining edge (of any mark --
directed, bidirected, or uncertain) to at least one disease node** in the final PAG. This
means Stable-FCI found the symptom and that disease to be *not* independent given every
conditioning set tried -- i.e. a candidate direct or confounded association -- rather than
a spurious marginal correlation explained away by other variables (which is exactly what
gets removed during Stable-FAS / Possible-D-SEP).


In [ ]:
# ---------------------------------------------------------------------------
# Feature extraction: symptoms adjacent to at least one disease node in the final PAG
# ---------------------------------------------------------------------------
t0 = time.time()

selected_symptoms = set()
for (n1, n2, ep1, ep2) in final_edges_info:
    t1, t2 = node_type.get(n1), node_type.get(n2)
    if t1 == "disease" and t2 == "symptom":
        selected_symptoms.add(n2)
    elif t2 == "disease" and t1 == "symptom":
        selected_symptoms.add(n1)

selected_symptoms = sorted(selected_symptoms)
pd.DataFrame({"symptom": selected_symptoms}).to_csv(
    RUN_DIR / "stable_fci_selected_symptoms.csv", index=False)
mark_time("feature_extraction", t0)

print(f"Stable-FCI selected {len(selected_symptoms)} / {len(symptom_cols)} symptoms "
      f"({100 * len(selected_symptoms) / max(len(symptom_cols), 1):.1f}% retained, "
      f"{100 * (1 - len(selected_symptoms) / max(len(symptom_cols), 1)):.1f}% reduction).")

if len(selected_symptoms) == 0:
    warnings.warn(
        "No symptoms were connected to any disease node in the final PAG. This can happen "
        "with a small MAX_DEPTH, a strict ALPHA, or a small training set relative to the "
        "number of variables. Consider raising CONFIG['MAX_DEPTH'] / CONFIG['ALPHA']."
    )


## 12. Multiclass Logistic Regression — Stable-FCI Features vs. All Symptoms

Both prediction experiments use the **same stratified 80/20 train/test split**.

- **Stable-FCI branch:** only symptoms adjacent to disease nodes in the final Stable-FCI PAG.
- **No-FCI baseline:** all symptom variables.
- **Model:** multiclass Logistic Regression with **L2 regularization**.
- **Hyperparameter:** `C`, selected from `LR_C_GRID` using **5-fold stratified cross-validation on the training set only**.
- **Primary final evaluation:** the untouched test set.

The test set is never used to choose `C`, select features, or fit the final coefficients.

In [ ]:
# ---------------------------------------------------------------------------
# Prepare features and target for both prediction experiments
# ---------------------------------------------------------------------------
y_train = train_df["_disease_label"].astype(str)
y_test = test_df["_disease_label"].astype(str)

all_symptom_feature_cols = list(symptom_cols)
fci_feature_cols = list(selected_symptoms)

if len(fci_feature_cols) == 0:
    raise RuntimeError(
        "Stable-FCI produced no disease-adjacent symptoms. "
        "The prediction stage is stopped rather than falling back to all symptoms."
    )

X_train_fci = train_df[fci_feature_cols]
X_test_fci = test_df[fci_feature_cols]
X_train_all = train_df[all_symptom_feature_cols]
X_test_all = test_df[all_symptom_feature_cols]

print(f"Stable-FCI features: {X_train_fci.shape[1]} / {len(all_symptom_feature_cols)} symptoms")
print(f"All-symptom baseline: {X_train_all.shape[1]} symptoms")
print(f"Training patients: {len(y_train)} | Test patients: {len(y_test)}")


## 13. Overfitting Control — What We Apply

The prediction stage explicitly uses the safeguards we want for a reliable comparison:

1. **Held-out test set:** split once at the beginning; never used for model selection.
2. **Stratified 5-fold CV:** used only inside the training set to choose `C`.
3. **L2 regularization:** prevents unnecessarily large Logistic Regression coefficients.
4. **Same CV protocol for both branches:** makes Stable-FCI vs. No-FCI comparison fair.
5. **Train/test generalization gap:** reported as a diagnostic for possible overfitting.

Important methodological note: Stable-FCI feature selection is performed once using the full training split, so the final held-out test result remains protected from test leakage. A stricter nested analysis would repeat FCI feature selection inside each inner CV fold; that is a separate computationally expensive experiment and is not silently mixed into this main Stable-FCI result.

In [ ]:
# ---------------------------------------------------------------------------
# Logistic Regression factory
# ---------------------------------------------------------------------------
def make_logistic_model(C):
    """Create the same regularized multiclass Logistic Regression model for both branches."""
    return LogisticRegression(
        C=float(C),
        penalty=CONFIG["LR_PENALTY"],
        solver=CONFIG["LR_SOLVER"],
        max_iter=CONFIG["LR_MAX_ITER"],
        random_state=CONFIG["RANDOM_STATE"],
    )

print(
    "Logistic Regression configuration:",
    f"penalty={CONFIG['LR_PENALTY']}, solver={CONFIG['LR_SOLVER']},",
    f"C grid={CONFIG['LR_C_GRID']}, CV folds={CONFIG['LR_CV_FOLDS']}"
)


In [ ]:
# ---------------------------------------------------------------------------
# 5-fold stratified CV hyperparameter tuning
# ---------------------------------------------------------------------------
def tune_logistic_regression(X, y, branch_name):
    """Choose C using macro-F1 averaged across stratified CV folds on TRAIN only."""
    X_np = X.to_numpy(dtype=np.float64)
    y_np = np.asarray(y)

    skf = StratifiedKFold(
        n_splits=CONFIG["LR_CV_FOLDS"],
        shuffle=True,
        random_state=CONFIG["RANDOM_STATE"],
    )

    rows = []
    best_C = None
    best_mean_f1 = -np.inf

    print(f"\n{branch_name}: 5-fold CV for C")
    print("=" * 70)

    for C in CONFIG["LR_C_GRID"]:
        fold_scores = []

        for fold_number, (tr_idx, val_idx) in enumerate(skf.split(X_np, y_np), start=1):
            model = make_logistic_model(C)
            model.fit(X_np[tr_idx], y_np[tr_idx])
            val_pred = model.predict(X_np[val_idx])
            fold_f1 = f1_score(
                y_np[val_idx], val_pred, average="macro", zero_division=0
            )
            fold_scores.append(float(fold_f1))

        mean_f1 = float(np.mean(fold_scores))
        std_f1 = float(np.std(fold_scores, ddof=1)) if len(fold_scores) > 1 else 0.0

        rows.append({
            "branch": branch_name,
            "C": float(C),
            "mean_cv_macro_f1": mean_f1,
            "std_cv_macro_f1": std_f1,
            **{f"fold_{i}_macro_f1": score for i, score in enumerate(fold_scores, start=1)},
        })

        print(
            f"C={C:<8g} | mean macro-F1={mean_f1:.4f} | "
            f"std={std_f1:.4f} | folds={[round(s, 4) for s in fold_scores]}"
        )

        if mean_f1 > best_mean_f1:
            best_mean_f1 = mean_f1
            best_C = float(C)

    cv_df = pd.DataFrame(rows)
    print(f"\n{branch_name}: best C = {best_C} | best mean CV macro-F1 = {best_mean_f1:.4f}")
    return best_C, best_mean_f1, cv_df

fci_best_C, fci_best_cv_f1, fci_cv_df = tune_logistic_regression(
    X_train_fci, y_train, "Stable-FCI features"
)

all_best_C, all_best_cv_f1, all_cv_df = tune_logistic_regression(
    X_train_all, y_train, "All symptoms"
)

fci_cv_df.to_csv(LR_DIR / "stable_fci_logistic_cv_results.csv", index=False)
all_cv_df.to_csv(LR_DIR / "all_symptoms_logistic_cv_results.csv", index=False)


## 14. Fit Final Logistic Regression Models on TRAIN Only

After `C` is chosen by cross-validation, each model is re-fitted on the **complete training split** using its selected `C`.

No test observations are used during fitting.

In [ ]:
# ---------------------------------------------------------------------------
# Fit final models on the COMPLETE TRAINING split
# ---------------------------------------------------------------------------
t0 = time.time()
lr_fci = make_logistic_model(fci_best_C)
lr_fci.fit(X_train_fci, y_train)
mark_time("model_training_fci_features", t0)


t0 = time.time()
lr_all = make_logistic_model(all_best_C)
lr_all.fit(X_train_all, y_train)
mark_time("model_training_all_features", t0)

joblib.dump(lr_fci, LR_DIR / "stable_fci_logistic_regression.joblib")
joblib.dump(lr_all, LR_DIR / "all_symptoms_logistic_regression.joblib")

print("Final Logistic Regression models fitted on training data only.")


## 15. Evaluation on TRAIN and TEST

The train scores are diagnostics for checking the generalization gap. The **test scores are the primary final results** because the test set was kept untouched during training and `C` selection.

In [ ]:
# ---------------------------------------------------------------------------
# Evaluation helper for multiclass Logistic Regression
# ---------------------------------------------------------------------------
def evaluate_logistic_model(model, X, y, split_name, model_name, cm_path):
    t0 = time.time()
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)

    accuracy = accuracy_score(y, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y, y_pred, average="weighted", zero_division=0
    )

    labels = list(model.classes_)
    y_bin = label_binarize(y, classes=labels)
    try:
        roc_auc_macro = roc_auc_score(
            y_bin, y_proba, average="macro", multi_class="ovr"
        )
    except ValueError:
        roc_auc_macro = None

    try:
        top3_accuracy = top_k_accuracy_score(y, y_proba, k=3, labels=labels)
    except ValueError:
        top3_accuracy = None

    metrics = {
        "split": split_name,
        "model": model_name,
        "n_features": int(X.shape[1]),
        "accuracy": float(accuracy),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
        "roc_auc_macro_ovr": None if roc_auc_macro is None else float(roc_auc_macro),
        "top3_accuracy": None if top3_accuracy is None else float(top3_accuracy),
        "evaluation_time_sec": round(time.time() - t0, 4),
    }

    cm = confusion_matrix(y, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(14, 12))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, xticks_rotation=90, colorbar=True)
    ax.set_title(f"Confusion Matrix — {model_name} — {split_name}")
    fig.tight_layout()
    fig.savefig(cm_path, format="pdf", bbox_inches="tight")
    plt.close(fig)

    print(
        f"[{model_name} | {split_name}] "
        f"accuracy={accuracy:.4f} | macro-F1={f1_macro:.4f} | "
        f"weighted-F1={f1_weighted:.4f} | top-3={top3_accuracy}"
    )
    return metrics, y_pred, y_proba


fci_train_metrics, _, _ = evaluate_logistic_model(
    lr_fci, X_train_fci, y_train, "train", "Stable-FCI features",
    LR_DIR / "stable_fci_train_confusion_matrix.pdf"
)
fci_test_metrics, y_pred_fci, y_proba_fci = evaluate_logistic_model(
    lr_fci, X_test_fci, y_test, "test", "Stable-FCI features",
    LR_DIR / "stable_fci_test_confusion_matrix.pdf"
)

all_train_metrics, _, _ = evaluate_logistic_model(
    lr_all, X_train_all, y_train, "train", "All symptoms",
    LR_DIR / "all_symptoms_train_confusion_matrix.pdf"
)
all_test_metrics, y_pred_all, y_proba_all = evaluate_logistic_model(
    lr_all, X_test_all, y_test, "test", "All symptoms",
    LR_DIR / "all_symptoms_test_confusion_matrix.pdf"
)


In [ ]:
# ---------------------------------------------------------------------------
# Generalization gaps: TRAIN score - TEST score
# ---------------------------------------------------------------------------
def make_gap(train_metrics, test_metrics):
    metric_keys = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "precision_weighted", "recall_weighted", "f1_weighted",
        "roc_auc_macro_ovr", "top3_accuracy"
    ]
    gap = {}
    for key in metric_keys:
        train_value = train_metrics.get(key)
        test_value = test_metrics.get(key)
        gap[key] = None if train_value is None or test_value is None else float(train_value - test_value)
    return gap

fci_generalization_gap = make_gap(fci_train_metrics, fci_test_metrics)
all_generalization_gap = make_gap(all_train_metrics, all_test_metrics)

print("\nGENERALIZATION GAPS (TRAIN - TEST)")
print("=" * 75)
for key in ["accuracy", "f1_macro", "f1_weighted", "top3_accuracy"]:
    print(
        f"{key:<20s} | Stable-FCI: {fci_generalization_gap[key]:+.4f} "
        f"| All symptoms: {all_generalization_gap[key]:+.4f}"
    )


### Metrics Bar Chart — Logistic Regression

`plot_fci_vs_all_barplot` is generic (it only reads keys out of a test-metrics
dict), so it is defined once here and reused for Random Forest and XGBoost
below without redefinition.


In [ ]:
# ---------------------------------------------------------------------------
# Reusable Stable-FCI-vs-All-Symptoms metrics bar chart (used for every model)
# ---------------------------------------------------------------------------
def plot_fci_vs_all_barplot(model_label, fci_metrics, all_metrics, n_fci, n_all, save_path):
    metric_specs = [
        ("accuracy", "Accuracy"),
        ("precision_macro", "Precision (macro)"),
        ("recall_macro", "Recall (macro)"),
        ("f1_macro", "F1 (macro)"),
        ("roc_auc_macro_ovr", "ROC-AUC (macro, OvR)"),
        ("top3_accuracy", "Top-3 Accuracy"),
    ]
    labels = [disp for _, disp in metric_specs]
    fci_vals = [fci_metrics[key] for key, _ in metric_specs]
    all_vals = [all_metrics[key] for key, _ in metric_specs]

    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(14, 7))
    bars1 = ax.bar(x - width / 2, fci_vals, width,
                    label=f"Stable-FCI features (n={n_fci})", color="#1f9e89")
    bars2 = ax.bar(x + width / 2, all_vals, width,
                    label=f"All features (n={n_all})", color="#f0653f")

    for bars in (bars1, bars2):
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f"{h:.3f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points", ha="center", fontsize=10)

    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.08)
    ax.set_title(f"{model_label}: Stable-FCI-Selected Features vs. All Symptoms")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.grid(True, axis="y", alpha=0.3, linestyle="--")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(save_path, format="pdf", bbox_inches="tight")
    plt.show()


n_fci, n_all = len(fci_feature_cols), len(all_symptom_feature_cols)

plot_fci_vs_all_barplot(
    "Logistic Regression", fci_test_metrics, all_test_metrics,
    n_fci, n_all, LR_DIR / "fci_vs_all_barplot.pdf"
)


## 16. Stable-FCI vs. No-FCI — Primary TEST Comparison

In [ ]:
# ---------------------------------------------------------------------------
# Primary comparison on the untouched TEST set
# ---------------------------------------------------------------------------
comparison_metrics = [
    "accuracy", "precision_macro", "recall_macro", "f1_macro",
    "precision_weighted", "recall_weighted", "f1_weighted",
    "roc_auc_macro_ovr", "top3_accuracy"
]

comparison_rows = []
for key in comparison_metrics:
    comparison_rows.append({
        "metric": key,
        "stable_fci_test": fci_test_metrics[key],
        "all_symptoms_test": all_test_metrics[key],
        "stable_fci_minus_all": (
            None if fci_test_metrics[key] is None or all_test_metrics[key] is None
            else fci_test_metrics[key] - all_test_metrics[key]
        ),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(LR_DIR / "fci_vs_no_fci_test_comparison.csv", index=False)

print("STABLE-FCI vs NO-FCI — TEST SET")
print("=" * 95)
print(comparison_df.to_string(index=False))


## 17. Train/Test Generalization Comparison

In [ ]:
# ---------------------------------------------------------------------------
# Train vs test comparison for overfitting diagnostics
# ---------------------------------------------------------------------------
generalization_df = pd.DataFrame({
    "metric": comparison_metrics,
    "stable_fci_train": [fci_train_metrics[k] for k in comparison_metrics],
    "stable_fci_test": [fci_test_metrics[k] for k in comparison_metrics],
    "stable_fci_gap": [fci_generalization_gap[k] for k in comparison_metrics],
    "all_symptoms_train": [all_train_metrics[k] for k in comparison_metrics],
    "all_symptoms_test": [all_test_metrics[k] for k in comparison_metrics],
    "all_symptoms_gap": [all_generalization_gap[k] for k in comparison_metrics],
})

generalization_df.to_csv(LR_DIR / "train_test_generalization_comparison.csv", index=False)
print(generalization_df.to_string(index=False))


## 18. Coefficient Inspection

In [ ]:
# ---------------------------------------------------------------------------
# Save learned Logistic Regression coefficients
# ---------------------------------------------------------------------------
coef_rows = []
for model_name, model, feat_cols in [
    ("stable_fci_features", lr_fci, fci_feature_cols),
    ("all_symptoms", lr_all, all_symptom_feature_cols),
]:
    for class_name, row in zip(model.classes_, model.coef_):
        for feature, coefficient in zip(feat_cols, row):
            coef_rows.append({
                "model": model_name,
                "disease_class": class_name,
                "feature": feature,
                "coefficient": float(coefficient),
                "absolute_coefficient": float(abs(coefficient)),
            })

coefficients_df = pd.DataFrame(coef_rows)
coefficients_df.to_csv(LR_DIR / "logistic_regression_coefficients.csv", index=False)
print(f"Saved {len(coefficients_df)} coefficient rows.")


## 19. C Tuning / Regularization Results

In [ ]:
# ---------------------------------------------------------------------------
# Plot mean CV macro-F1 against C for both branches
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogx(fci_cv_df["C"], fci_cv_df["mean_cv_macro_f1"], marker="o", label="Stable-FCI features")
ax.semilogx(all_cv_df["C"], all_cv_df["mean_cv_macro_f1"], marker="o", label="All symptoms")
ax.set_xlabel("C (inverse regularization strength)")
ax.set_ylabel("Mean 5-fold CV macro-F1")
ax.set_title("Logistic Regression C Tuning")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(LR_DIR / "logistic_C_tuning.pdf", format="pdf", bbox_inches="tight")
plt.show()


## 20. Final Summary and Reproducibility

In [ ]:
# ---------------------------------------------------------------------------
# Save summary artifacts for this run
# ---------------------------------------------------------------------------
model_summary = {
    "stable_fci": {
        "n_features": len(fci_feature_cols),
        "best_C": fci_best_C,
        "best_cv_macro_f1": fci_best_cv_f1,
        "train_metrics": fci_train_metrics,
        "test_metrics": fci_test_metrics,
        "generalization_gap": fci_generalization_gap,
    },
    "all_symptoms": {
        "n_features": len(all_symptom_feature_cols),
        "best_C": all_best_C,
        "best_cv_macro_f1": all_best_cv_f1,
        "train_metrics": all_train_metrics,
        "test_metrics": all_test_metrics,
        "generalization_gap": all_generalization_gap,
    },
    "feature_reduction_pct": round(
        100.0 * (1.0 - len(fci_feature_cols) / len(all_symptom_feature_cols)), 2
    ),
    "overfitting_controls": {
        "stratified_train_test_split": True,
        "test_size": CONFIG["TEST_SIZE"],
        "random_state": CONFIG["RANDOM_STATE"],
        "stable_fci_run_on_training_only": True,
        "stratified_cv_folds": CONFIG["LR_CV_FOLDS"],
        "regularization": CONFIG["LR_PENALTY"],
        "C_grid": CONFIG["LR_C_GRID"],
        "CV_selection_metric": CONFIG["LR_SCORING"],
        "test_used_for_C_selection": False,
    },
}

save_json(model_summary, RUN_DIR / "logistic_regression_summary.json")
save_json({
    "stable_fci_cv": fci_cv_df.to_dict(orient="records"),
    "all_symptoms_cv": all_cv_df.to_dict(orient="records"),
}, LR_DIR / "logistic_regression_cv_results.json")

print("=" * 80)
print("FINAL LOGISTIC REGRESSION SUMMARY")
print("=" * 80)
print(f"Stable-FCI selected features : {len(fci_feature_cols)} / {len(all_symptom_feature_cols)}")
print(f"Stable-FCI best C            : {fci_best_C}")
print(f"Stable-FCI CV macro-F1       : {fci_best_cv_f1:.4f}")
print(f"Stable-FCI TEST macro-F1     : {fci_test_metrics['f1_macro']:.4f}")
print(f"All-symptoms best C          : {all_best_C}")
print(f"All-symptoms CV macro-F1     : {all_best_cv_f1:.4f}")
print(f"All-symptoms TEST macro-F1   : {all_test_metrics['f1_macro']:.4f}")
print(f"Feature reduction            : {model_summary['feature_reduction_pct']:.2f}%")
print(f"Results saved under          : {RUN_DIR.resolve()}")


## 22. Random Forest — Stable-FCI Features vs. All Symptoms

Same comparison as Section 12-20, same train/test split, same `X_train_fci` /
`X_test_fci` / `X_train_all` / `X_test_all` / `y_train` / `y_test` prepared in
Section 12 — only the model changes. `evaluate_logistic_model` and `make_gap`
(Sections 15 and 17) are model-agnostic (they only call `.predict` /
`.predict_proba` / `.classes_`), so they are reused as-is for Random Forest
rather than duplicated.


In [ ]:
# ---------------------------------------------------------------------------
# Random Forest factory
# ---------------------------------------------------------------------------
def make_random_forest_model(params):
    """Create the same-shaped Random Forest model for both branches."""
    return RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_leaf=CONFIG["RF_MIN_SAMPLES_LEAF"],
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=CONFIG["RF_N_JOBS"],
    )

print(
    "Random Forest configuration:",
    f"min_samples_leaf={CONFIG['RF_MIN_SAMPLES_LEAF']},",
    f"param grid={CONFIG['RF_PARAM_GRID']}, CV folds={CONFIG['RF_CV_FOLDS']}"
)


In [ ]:
# ---------------------------------------------------------------------------
# 5-fold stratified CV hyperparameter tuning (n_estimators, max_depth)
# ---------------------------------------------------------------------------
def tune_random_forest(X, y, branch_name):
    """Choose (n_estimators, max_depth) using macro-F1 averaged across
    stratified CV folds on TRAIN only -- mirrors tune_logistic_regression."""
    X_np = X.to_numpy(dtype=np.float64)
    y_np = np.asarray(y)

    skf = StratifiedKFold(
        n_splits=CONFIG["RF_CV_FOLDS"],
        shuffle=True,
        random_state=CONFIG["RANDOM_STATE"],
    )

    rows = []
    best_params = None
    best_mean_f1 = -np.inf

    print(f"\n{branch_name}: 5-fold CV for (n_estimators, max_depth)")
    print("=" * 70)

    for params in CONFIG["RF_PARAM_GRID"]:
        fold_scores = []

        for fold_number, (tr_idx, val_idx) in enumerate(skf.split(X_np, y_np), start=1):
            model = make_random_forest_model(params)
            model.fit(X_np[tr_idx], y_np[tr_idx])
            val_pred = model.predict(X_np[val_idx])
            fold_f1 = f1_score(
                y_np[val_idx], val_pred, average="macro", zero_division=0
            )
            fold_scores.append(float(fold_f1))

        mean_f1 = float(np.mean(fold_scores))
        std_f1 = float(np.std(fold_scores, ddof=1)) if len(fold_scores) > 1 else 0.0

        rows.append({
            "branch": branch_name,
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "mean_cv_macro_f1": mean_f1,
            "std_cv_macro_f1": std_f1,
            **{f"fold_{i}_macro_f1": score for i, score in enumerate(fold_scores, start=1)},
        })

        print(
            f"n_estimators={params['n_estimators']:<5} max_depth={str(params['max_depth']):<5} | "
            f"mean macro-F1={mean_f1:.4f} | std={std_f1:.4f} | "
            f"folds={[round(s, 4) for s in fold_scores]}"
        )

        if mean_f1 > best_mean_f1:
            best_mean_f1 = mean_f1
            best_params = dict(params)

    cv_df = pd.DataFrame(rows)
    print(f"\n{branch_name}: best params = {best_params} | best mean CV macro-F1 = {best_mean_f1:.4f}")
    return best_params, best_mean_f1, cv_df

rf_fci_best_params, rf_fci_best_cv_f1, rf_fci_cv_df = tune_random_forest(
    X_train_fci, y_train, "Stable-FCI features"
)

rf_all_best_params, rf_all_best_cv_f1, rf_all_cv_df = tune_random_forest(
    X_train_all, y_train, "All symptoms"
)

rf_fci_cv_df.to_csv(RF_DIR / "stable_fci_random_forest_cv_results.csv", index=False)
rf_all_cv_df.to_csv(RF_DIR / "all_symptoms_random_forest_cv_results.csv", index=False)


## 23. Fit Final Random Forest Models on TRAIN Only

After `(n_estimators, max_depth)` is chosen by cross-validation, each model is
re-fitted on the complete training split using its selected params. No test
observations are used here, matching Section 14 for Logistic Regression.


In [ ]:
# ---------------------------------------------------------------------------
# Fit final Random Forest models on the COMPLETE TRAINING split
# ---------------------------------------------------------------------------
t0 = time.time()
rf_fci = make_random_forest_model(rf_fci_best_params)
rf_fci.fit(X_train_fci, y_train)
mark_time("model_training_rf_fci_features", t0)

t0 = time.time()
rf_all = make_random_forest_model(rf_all_best_params)
rf_all.fit(X_train_all, y_train)
mark_time("model_training_rf_all_features", t0)

joblib.dump(rf_fci, RF_DIR / "stable_fci_random_forest.joblib")
joblib.dump(rf_all, RF_DIR / "all_symptoms_random_forest.joblib")

print("Final Random Forest models fitted on training data only.")


## 24. Evaluation on TRAIN and TEST — Random Forest

Reuses `evaluate_logistic_model` from Section 15 (it is model-agnostic) so the
metric definitions are identical to the Logistic Regression branch.


In [ ]:
# ---------------------------------------------------------------------------
# Evaluate Random Forest models (reuses evaluate_logistic_model, Section 15)
# ---------------------------------------------------------------------------
rf_fci_train_metrics, _, _ = evaluate_logistic_model(
    rf_fci, X_train_fci, y_train, "train", "Random Forest - Stable-FCI features",
    RF_DIR / "stable_fci_train_confusion_matrix.pdf"
)
rf_fci_test_metrics, y_pred_rf_fci, y_proba_rf_fci = evaluate_logistic_model(
    rf_fci, X_test_fci, y_test, "test", "Random Forest - Stable-FCI features",
    RF_DIR / "stable_fci_test_confusion_matrix.pdf"
)

rf_all_train_metrics, _, _ = evaluate_logistic_model(
    rf_all, X_train_all, y_train, "train", "Random Forest - All symptoms",
    RF_DIR / "all_symptoms_train_confusion_matrix.pdf"
)
rf_all_test_metrics, y_pred_rf_all, y_proba_rf_all = evaluate_logistic_model(
    rf_all, X_test_all, y_test, "test", "Random Forest - All symptoms",
    RF_DIR / "all_symptoms_test_confusion_matrix.pdf"
)


In [ ]:
# ---------------------------------------------------------------------------
# Generalization gaps: TRAIN score - TEST score (reuses make_gap, Section 17)
# ---------------------------------------------------------------------------
rf_fci_generalization_gap = make_gap(rf_fci_train_metrics, rf_fci_test_metrics)
rf_all_generalization_gap = make_gap(rf_all_train_metrics, rf_all_test_metrics)

print("\nGENERALIZATION GAPS (TRAIN - TEST) -- Random Forest")
print("=" * 75)
for key in ["accuracy", "f1_macro", "f1_weighted", "top3_accuracy"]:
    print(
        f"{key:<20s} | Stable-FCI: {rf_fci_generalization_gap[key]:+.4f} "
        f"| All symptoms: {rf_all_generalization_gap[key]:+.4f}"
    )


### Metrics Bar Chart — Random Forest


In [ ]:
# ---------------------------------------------------------------------------
# Stable-FCI vs. All Symptoms metrics bar chart -- Random Forest
# ---------------------------------------------------------------------------
plot_fci_vs_all_barplot(
    "Random Forest", rf_fci_test_metrics, rf_all_test_metrics,
    n_fci, n_all, RF_DIR / "fci_vs_all_barplot.pdf"
)


## 25. Stable-FCI vs. No-FCI — Random Forest TEST Comparison


In [ ]:
# ---------------------------------------------------------------------------
# Primary Random Forest comparison on the untouched TEST set
# ---------------------------------------------------------------------------
rf_comparison_rows = []
for key in comparison_metrics:
    rf_comparison_rows.append({
        "metric": key,
        "stable_fci_test": rf_fci_test_metrics[key],
        "all_symptoms_test": rf_all_test_metrics[key],
        "stable_fci_minus_all": (
            None if rf_fci_test_metrics[key] is None or rf_all_test_metrics[key] is None
            else rf_fci_test_metrics[key] - rf_all_test_metrics[key]
        ),
    })

rf_comparison_df = pd.DataFrame(rf_comparison_rows)
rf_comparison_df.to_csv(RF_DIR / "fci_vs_no_fci_test_comparison.csv", index=False)

print("RANDOM FOREST -- STABLE-FCI vs NO-FCI -- TEST SET")
print("=" * 95)
print(rf_comparison_df.to_string(index=False))


## 26. Train/Test Generalization Comparison — Random Forest


In [ ]:
# ---------------------------------------------------------------------------
# Random Forest train vs test comparison for overfitting diagnostics
# ---------------------------------------------------------------------------
rf_generalization_df = pd.DataFrame({
    "metric": comparison_metrics,
    "stable_fci_train": [rf_fci_train_metrics[k] for k in comparison_metrics],
    "stable_fci_test": [rf_fci_test_metrics[k] for k in comparison_metrics],
    "stable_fci_gap": [rf_fci_generalization_gap[k] for k in comparison_metrics],
    "all_symptoms_train": [rf_all_train_metrics[k] for k in comparison_metrics],
    "all_symptoms_test": [rf_all_test_metrics[k] for k in comparison_metrics],
    "all_symptoms_gap": [rf_all_generalization_gap[k] for k in comparison_metrics],
})

rf_generalization_df.to_csv(RF_DIR / "train_test_generalization_comparison.csv", index=False)
print(rf_generalization_df.to_string(index=False))


## 27. Feature Importance Inspection — Random Forest

Random Forest has one importance value per feature (not per class, unlike the
Logistic Regression coefficients in Section 18).


In [ ]:
# ---------------------------------------------------------------------------
# Save learned Random Forest feature importances
# ---------------------------------------------------------------------------
rf_importance_rows = []
for model_name, model, feat_cols in [
    ("stable_fci_features", rf_fci, fci_feature_cols),
    ("all_symptoms", rf_all, all_symptom_feature_cols),
]:
    for feature, importance in zip(feat_cols, model.feature_importances_):
        rf_importance_rows.append({
            "model": model_name,
            "feature": feature,
            "importance": float(importance),
        })

rf_importances_df = pd.DataFrame(rf_importance_rows)
rf_importances_df.to_csv(RF_DIR / "random_forest_feature_importances.csv", index=False)
print(f"Saved {len(rf_importances_df)} feature-importance rows.")

print("\nTop 10 features -- Stable-FCI branch:")
print(
    rf_importances_df[rf_importances_df["model"] == "stable_fci_features"]
    .sort_values("importance", ascending=False)
    .head(10)
    .to_string(index=False)
)


## 28. Random Forest Hyperparameter Tuning Results


In [ ]:
# ---------------------------------------------------------------------------
# Plot mean CV macro-F1 for each (n_estimators, max_depth) combo, both branches
# ---------------------------------------------------------------------------
def _combo_label(row):
    depth = "None" if row["max_depth"] is None else row["max_depth"]
    return f"n={int(row['n_estimators'])}, depth={depth}"

rf_fci_cv_df["combo"] = rf_fci_cv_df.apply(_combo_label, axis=1)
rf_all_cv_df["combo"] = rf_all_cv_df.apply(_combo_label, axis=1)

combos = rf_fci_cv_df["combo"].tolist()
x = np.arange(len(combos))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width / 2, rf_fci_cv_df["mean_cv_macro_f1"], width, label="Stable-FCI features")
ax.bar(x + width / 2, rf_all_cv_df["mean_cv_macro_f1"], width, label="All symptoms")
ax.set_xticks(x)
ax.set_xticklabels(combos, rotation=45, ha="right")
ax.set_ylabel("Mean 5-fold CV macro-F1")
ax.set_title("Random Forest Hyperparameter Tuning")
ax.grid(True, axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(RF_DIR / "random_forest_tuning.pdf", format="pdf", bbox_inches="tight")
plt.show()


## 29. Random Forest Final Summary


In [ ]:
# ---------------------------------------------------------------------------
# Save Random Forest summary artifacts for this run
# ---------------------------------------------------------------------------
rf_model_summary = {
    "stable_fci": {
        "n_features": len(fci_feature_cols),
        "best_params": rf_fci_best_params,
        "best_cv_macro_f1": rf_fci_best_cv_f1,
        "train_metrics": rf_fci_train_metrics,
        "test_metrics": rf_fci_test_metrics,
        "generalization_gap": rf_fci_generalization_gap,
    },
    "all_symptoms": {
        "n_features": len(all_symptom_feature_cols),
        "best_params": rf_all_best_params,
        "best_cv_macro_f1": rf_all_best_cv_f1,
        "train_metrics": rf_all_train_metrics,
        "test_metrics": rf_all_test_metrics,
        "generalization_gap": rf_all_generalization_gap,
    },
    "feature_reduction_pct": round(
        100.0 * (1.0 - len(fci_feature_cols) / len(all_symptom_feature_cols)), 2
    ),
    "overfitting_controls": {
        "stratified_train_test_split": True,
        "test_size": CONFIG["TEST_SIZE"],
        "random_state": CONFIG["RANDOM_STATE"],
        "stable_fci_run_on_training_only": True,
        "stratified_cv_folds": CONFIG["RF_CV_FOLDS"],
        "min_samples_leaf": CONFIG["RF_MIN_SAMPLES_LEAF"],
        "param_grid": CONFIG["RF_PARAM_GRID"],
        "CV_selection_metric": CONFIG["RF_SCORING"],
        "test_used_for_param_selection": False,
    },
}

save_json(rf_model_summary, RUN_DIR / "random_forest_summary.json")
save_json({
    "stable_fci_cv": rf_fci_cv_df.to_dict(orient="records"),
    "all_symptoms_cv": rf_all_cv_df.to_dict(orient="records"),
}, RF_DIR / "random_forest_cv_results.json")

print("=" * 80)
print("FINAL RANDOM FOREST SUMMARY")
print("=" * 80)
print(f"Stable-FCI selected features : {len(fci_feature_cols)} / {len(all_symptom_feature_cols)}")
print(f"Stable-FCI best params        : {rf_fci_best_params}")
print(f"Stable-FCI CV macro-F1        : {rf_fci_best_cv_f1:.4f}")
print(f"Stable-FCI TEST macro-F1      : {rf_fci_test_metrics['f1_macro']:.4f}")
print(f"All-symptoms best params      : {rf_all_best_params}")
print(f"All-symptoms CV macro-F1      : {rf_all_best_cv_f1:.4f}")
print(f"All-symptoms TEST macro-F1    : {rf_all_test_metrics['f1_macro']:.4f}")
print(f"Feature reduction             : {rf_model_summary['feature_reduction_pct']:.2f}%")
print(f"Results saved under           : {RUN_DIR.resolve()}")


## 31. XGBoost — Stable-FCI Features vs. All Symptoms

Same comparison as Sections 12-20 and 22-29, same train/test split, same
`X_train_fci` / `X_test_fci` / `X_train_all` / `X_test_all` / `y_train` /
`y_test` prepared in Section 12 -- only the model changes. `XGBClassifier`
(xgboost >= 1.6) accepts the string disease labels directly and exposes
`.classes_`, so `evaluate_logistic_model` and `make_gap` are reused as-is,
exactly as they were for Random Forest.


In [ ]:
# ---------------------------------------------------------------------------
# XGBoost needs integer labels (0..n_classes-1), not raw strings. Encode once
# with a shared LabelEncoder so every fold/model uses the same mapping, and
# wrap XGBClassifier so it drops into evaluate_logistic_model exactly like
# LogisticRegression / RandomForestClassifier (fit on strings, predict strings,
# expose .classes_).
# ---------------------------------------------------------------------------
from sklearn.preprocessing import LabelEncoder

XGB_LABEL_ENCODER = LabelEncoder()
XGB_LABEL_ENCODER.fit(y_train)  # fit once on the full training label set

class XGBClassifierWithLabels:
    def __init__(self, **xgb_kwargs):
        self._model = XGBClassifier(**xgb_kwargs)
        self.classes_ = XGB_LABEL_ENCODER.classes_

    def fit(self, X, y):
        y_enc = XGB_LABEL_ENCODER.transform(y)
        self._model.fit(X, y_enc)
        return self

    def predict(self, X):
        pred_enc = self._model.predict(X)
        return XGB_LABEL_ENCODER.inverse_transform(pred_enc)

    def predict_proba(self, X):
        return self._model.predict_proba(X)

    @property
    def feature_importances_(self):
        return self._model.feature_importances_

In [ ]:
# ---------------------------------------------------------------------------
# XGBoost factory
# ---------------------------------------------------------------------------
def make_xgboost_model(params):
    """Create the same-shaped XGBoost model for both branches."""
    return XGBClassifierWithLabels(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=CONFIG["XGB_SUBSAMPLE"],
        colsample_bytree=CONFIG["XGB_COLSAMPLE_BYTREE"],
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=CONFIG["XGB_N_JOBS"],
    )

print(
    "XGBoost configuration:",
    f"subsample={CONFIG['XGB_SUBSAMPLE']}, colsample_bytree={CONFIG['XGB_COLSAMPLE_BYTREE']},",
    f"param grid={CONFIG['XGB_PARAM_GRID']}, CV folds={CONFIG['XGB_CV_FOLDS']}"
)

In [ ]:
# ---------------------------------------------------------------------------
# 5-fold stratified CV hyperparameter tuning (n_estimators, max_depth, learning_rate)
# ---------------------------------------------------------------------------
def tune_xgboost(X, y, branch_name):
    """Choose (n_estimators, max_depth, learning_rate) using macro-F1 averaged
    across stratified CV folds on TRAIN only -- mirrors tune_random_forest."""
    X_np = X.to_numpy(dtype=np.float64)
    y_np = np.asarray(y)

    skf = StratifiedKFold(
        n_splits=CONFIG["XGB_CV_FOLDS"],
        shuffle=True,
        random_state=CONFIG["RANDOM_STATE"],
    )

    rows = []
    best_params = None
    best_mean_f1 = -np.inf

    print(f"\n{branch_name}: 5-fold CV for (n_estimators, max_depth, learning_rate)")
    print("=" * 70)

    for params in CONFIG["XGB_PARAM_GRID"]:
        fold_scores = []

        for fold_number, (tr_idx, val_idx) in enumerate(skf.split(X_np, y_np), start=1):
            model = make_xgboost_model(params)
            model.fit(X_np[tr_idx], y_np[tr_idx])
            val_pred = model.predict(X_np[val_idx])
            fold_f1 = f1_score(
                y_np[val_idx], val_pred, average="macro", zero_division=0
            )
            fold_scores.append(float(fold_f1))

        mean_f1 = float(np.mean(fold_scores))
        std_f1 = float(np.std(fold_scores, ddof=1)) if len(fold_scores) > 1 else 0.0

        rows.append({
            "branch": branch_name,
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "learning_rate": params["learning_rate"],
            "mean_cv_macro_f1": mean_f1,
            "std_cv_macro_f1": std_f1,
            **{f"fold_{i}_macro_f1": score for i, score in enumerate(fold_scores, start=1)},
        })

        print(
            f"n_estimators={params['n_estimators']:<5} max_depth={params['max_depth']:<3} "
            f"lr={params['learning_rate']:<6} | mean macro-F1={mean_f1:.4f} | "
            f"std={std_f1:.4f} | folds={[round(s, 4) for s in fold_scores]}"
        )

        if mean_f1 > best_mean_f1:
            best_mean_f1 = mean_f1
            best_params = dict(params)

    cv_df = pd.DataFrame(rows)
    print(f"\n{branch_name}: best params = {best_params} | best mean CV macro-F1 = {best_mean_f1:.4f}")
    return best_params, best_mean_f1, cv_df

xgb_fci_best_params, xgb_fci_best_cv_f1, xgb_fci_cv_df = tune_xgboost(
    X_train_fci, y_train, "Stable-FCI features"
)

xgb_all_best_params, xgb_all_best_cv_f1, xgb_all_cv_df = tune_xgboost(
    X_train_all, y_train, "All symptoms"
)

xgb_fci_cv_df.to_csv(XGB_DIR / "stable_fci_xgboost_cv_results.csv", index=False)
xgb_all_cv_df.to_csv(XGB_DIR / "all_symptoms_xgboost_cv_results.csv", index=False)


## 32. Fit Final XGBoost Models on TRAIN Only

After `(n_estimators, max_depth, learning_rate)` is chosen by cross-validation,
each model is re-fitted on the complete training split using its selected
params. No test observations are used here, matching Sections 14 and 23.


In [ ]:
# ---------------------------------------------------------------------------
# Fit final XGBoost models on the COMPLETE TRAINING split
# ---------------------------------------------------------------------------
t0 = time.time()
xgb_fci = make_xgboost_model(xgb_fci_best_params)
xgb_fci.fit(X_train_fci, y_train)
mark_time("model_training_xgb_fci_features", t0)

t0 = time.time()
xgb_all = make_xgboost_model(xgb_all_best_params)
xgb_all.fit(X_train_all, y_train)
mark_time("model_training_xgb_all_features", t0)

joblib.dump(xgb_fci, XGB_DIR / "stable_fci_xgboost.joblib")
joblib.dump(xgb_all, XGB_DIR / "all_symptoms_xgboost.joblib")

print("Final XGBoost models fitted on training data only.")


## 33. Evaluation on TRAIN and TEST — XGBoost

Reuses `evaluate_logistic_model` from Section 15 (model-agnostic), same as
Random Forest in Section 24.


In [ ]:
# ---------------------------------------------------------------------------
# Evaluate XGBoost models (reuses evaluate_logistic_model, Section 15)
# ---------------------------------------------------------------------------
xgb_fci_train_metrics, _, _ = evaluate_logistic_model(
    xgb_fci, X_train_fci, y_train, "train", "XGBoost - Stable-FCI features",
    XGB_DIR / "stable_fci_train_confusion_matrix.pdf"
)
xgb_fci_test_metrics, y_pred_xgb_fci, y_proba_xgb_fci = evaluate_logistic_model(
    xgb_fci, X_test_fci, y_test, "test", "XGBoost - Stable-FCI features",
    XGB_DIR / "stable_fci_test_confusion_matrix.pdf"
)

xgb_all_train_metrics, _, _ = evaluate_logistic_model(
    xgb_all, X_train_all, y_train, "train", "XGBoost - All symptoms",
    XGB_DIR / "all_symptoms_train_confusion_matrix.pdf"
)
xgb_all_test_metrics, y_pred_xgb_all, y_proba_xgb_all = evaluate_logistic_model(
    xgb_all, X_test_all, y_test, "test", "XGBoost - All symptoms",
    XGB_DIR / "all_symptoms_test_confusion_matrix.pdf"
)


In [ ]:
# ---------------------------------------------------------------------------
# Generalization gaps: TRAIN score - TEST score (reuses make_gap, Section 17)
# ---------------------------------------------------------------------------
xgb_fci_generalization_gap = make_gap(xgb_fci_train_metrics, xgb_fci_test_metrics)
xgb_all_generalization_gap = make_gap(xgb_all_train_metrics, xgb_all_test_metrics)

print("\nGENERALIZATION GAPS (TRAIN - TEST) -- XGBoost")
print("=" * 75)
for key in ["accuracy", "f1_macro", "f1_weighted", "top3_accuracy"]:
    print(
        f"{key:<20s} | Stable-FCI: {xgb_fci_generalization_gap[key]:+.4f} "
        f"| All symptoms: {xgb_all_generalization_gap[key]:+.4f}"
    )


### Metrics Bar Chart — XGBoost


In [ ]:
# ---------------------------------------------------------------------------
# Stable-FCI vs. All Symptoms metrics bar chart -- XGBoost
# ---------------------------------------------------------------------------
plot_fci_vs_all_barplot(
    "XGBoost", xgb_fci_test_metrics, xgb_all_test_metrics,
    n_fci, n_all, XGB_DIR / "fci_vs_all_barplot.pdf"
)


## 34. Stable-FCI vs. No-FCI — XGBoost TEST Comparison


In [ ]:
# ---------------------------------------------------------------------------
# Primary XGBoost comparison on the untouched TEST set
# ---------------------------------------------------------------------------
xgb_comparison_rows = []
for key in comparison_metrics:
    xgb_comparison_rows.append({
        "metric": key,
        "stable_fci_test": xgb_fci_test_metrics[key],
        "all_symptoms_test": xgb_all_test_metrics[key],
        "stable_fci_minus_all": (
            None if xgb_fci_test_metrics[key] is None or xgb_all_test_metrics[key] is None
            else xgb_fci_test_metrics[key] - xgb_all_test_metrics[key]
        ),
    })

xgb_comparison_df = pd.DataFrame(xgb_comparison_rows)
xgb_comparison_df.to_csv(XGB_DIR / "fci_vs_no_fci_test_comparison.csv", index=False)

print("XGBOOST -- STABLE-FCI vs NO-FCI -- TEST SET")
print("=" * 95)
print(xgb_comparison_df.to_string(index=False))


## 35. Train/Test Generalization Comparison — XGBoost


In [ ]:
# ---------------------------------------------------------------------------
# XGBoost train vs test comparison for overfitting diagnostics
# ---------------------------------------------------------------------------
xgb_generalization_df = pd.DataFrame({
    "metric": comparison_metrics,
    "stable_fci_train": [xgb_fci_train_metrics[k] for k in comparison_metrics],
    "stable_fci_test": [xgb_fci_test_metrics[k] for k in comparison_metrics],
    "stable_fci_gap": [xgb_fci_generalization_gap[k] for k in comparison_metrics],
    "all_symptoms_train": [xgb_all_train_metrics[k] for k in comparison_metrics],
    "all_symptoms_test": [xgb_all_test_metrics[k] for k in comparison_metrics],
    "all_symptoms_gap": [xgb_all_generalization_gap[k] for k in comparison_metrics],
})

xgb_generalization_df.to_csv(XGB_DIR / "train_test_generalization_comparison.csv", index=False)
print(xgb_generalization_df.to_string(index=False))


## 36. Feature Importance Inspection — XGBoost

Uses XGBoost's built-in gain-based `.feature_importances_`, one value per
feature -- same shape as the Random Forest importances in Section 27.


In [ ]:
# ---------------------------------------------------------------------------
# Save learned XGBoost feature importances
# ---------------------------------------------------------------------------
xgb_importance_rows = []
for model_name, model, feat_cols in [
    ("stable_fci_features", xgb_fci, fci_feature_cols),
    ("all_symptoms", xgb_all, all_symptom_feature_cols),
]:
    for feature, importance in zip(feat_cols, model.feature_importances_):
        xgb_importance_rows.append({
            "model": model_name,
            "feature": feature,
            "importance": float(importance),
        })

xgb_importances_df = pd.DataFrame(xgb_importance_rows)
xgb_importances_df.to_csv(XGB_DIR / "xgboost_feature_importances.csv", index=False)
print(f"Saved {len(xgb_importances_df)} feature-importance rows.")

print("\nTop 10 features -- Stable-FCI branch:")
print(
    xgb_importances_df[xgb_importances_df["model"] == "stable_fci_features"]
    .sort_values("importance", ascending=False)
    .head(10)
    .to_string(index=False)
)


## 37. XGBoost Hyperparameter Tuning Results


In [ ]:
# ---------------------------------------------------------------------------
# Plot mean CV macro-F1 for each (n_estimators, max_depth, learning_rate)
# combo, both branches
# ---------------------------------------------------------------------------
def _xgb_combo_label(row):
    return f"n={int(row['n_estimators'])}, d={int(row['max_depth'])}, lr={row['learning_rate']}"

xgb_fci_cv_df["combo"] = xgb_fci_cv_df.apply(_xgb_combo_label, axis=1)
xgb_all_cv_df["combo"] = xgb_all_cv_df.apply(_xgb_combo_label, axis=1)

combos = xgb_fci_cv_df["combo"].tolist()
x = np.arange(len(combos))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width / 2, xgb_fci_cv_df["mean_cv_macro_f1"], width, label="Stable-FCI features")
ax.bar(x + width / 2, xgb_all_cv_df["mean_cv_macro_f1"], width, label="All symptoms")
ax.set_xticks(x)
ax.set_xticklabels(combos, rotation=45, ha="right")
ax.set_ylabel("Mean 5-fold CV macro-F1")
ax.set_title("XGBoost Hyperparameter Tuning")
ax.grid(True, axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(XGB_DIR / "xgboost_tuning.pdf", format="pdf", bbox_inches="tight")
plt.show()


## 38. XGBoost Final Summary


In [ ]:
# ---------------------------------------------------------------------------
# Save XGBoost summary artifacts for this run
# ---------------------------------------------------------------------------
xgb_model_summary = {
    "stable_fci": {
        "n_features": len(fci_feature_cols),
        "best_params": xgb_fci_best_params,
        "best_cv_macro_f1": xgb_fci_best_cv_f1,
        "train_metrics": xgb_fci_train_metrics,
        "test_metrics": xgb_fci_test_metrics,
        "generalization_gap": xgb_fci_generalization_gap,
    },
    "all_symptoms": {
        "n_features": len(all_symptom_feature_cols),
        "best_params": xgb_all_best_params,
        "best_cv_macro_f1": xgb_all_best_cv_f1,
        "train_metrics": xgb_all_train_metrics,
        "test_metrics": xgb_all_test_metrics,
        "generalization_gap": xgb_all_generalization_gap,
    },
    "feature_reduction_pct": round(
        100.0 * (1.0 - len(fci_feature_cols) / len(all_symptom_feature_cols)), 2
    ),
    "overfitting_controls": {
        "stratified_train_test_split": True,
        "test_size": CONFIG["TEST_SIZE"],
        "random_state": CONFIG["RANDOM_STATE"],
        "stable_fci_run_on_training_only": True,
        "stratified_cv_folds": CONFIG["XGB_CV_FOLDS"],
        "subsample": CONFIG["XGB_SUBSAMPLE"],
        "colsample_bytree": CONFIG["XGB_COLSAMPLE_BYTREE"],
        "param_grid": CONFIG["XGB_PARAM_GRID"],
        "CV_selection_metric": CONFIG["XGB_SCORING"],
        "test_used_for_param_selection": False,
    },
}

save_json(xgb_model_summary, RUN_DIR / "xgboost_summary.json")
save_json({
    "stable_fci_cv": xgb_fci_cv_df.to_dict(orient="records"),
    "all_symptoms_cv": xgb_all_cv_df.to_dict(orient="records"),
}, XGB_DIR / "xgboost_cv_results.json")

print("=" * 80)
print("FINAL XGBOOST SUMMARY")
print("=" * 80)
print(f"Stable-FCI selected features : {len(fci_feature_cols)} / {len(all_symptom_feature_cols)}")
print(f"Stable-FCI best params        : {xgb_fci_best_params}")
print(f"Stable-FCI CV macro-F1        : {xgb_fci_best_cv_f1:.4f}")
print(f"Stable-FCI TEST macro-F1      : {xgb_fci_test_metrics['f1_macro']:.4f}")
print(f"All-symptoms best params      : {xgb_all_best_params}")
print(f"All-symptoms CV macro-F1      : {xgb_all_best_cv_f1:.4f}")
print(f"All-symptoms TEST macro-F1    : {xgb_all_test_metrics['f1_macro']:.4f}")
print(f"Feature reduction             : {xgb_model_summary['feature_reduction_pct']:.2f}%")
print(f"Results saved under           : {RUN_DIR.resolve()}")


## 39. Logistic Regression vs. Random Forest vs. XGBoost — Head-to-Head

All three models were tuned and evaluated on the exact same Stable-FCI /
all-symptom feature sets and the exact same train/test split, so their TEST
macro-F1 scores are directly comparable.


In [ ]:
# ---------------------------------------------------------------------------
# Combined LR vs RF vs XGBoost comparison on the untouched TEST set
# ---------------------------------------------------------------------------
model_head_to_head = pd.DataFrame([
    {"model": "Logistic Regression", "branch": "Stable-FCI", **{k: fci_test_metrics[k] for k in comparison_metrics}},
    {"model": "Logistic Regression", "branch": "All symptoms", **{k: all_test_metrics[k] for k in comparison_metrics}},
    {"model": "Random Forest", "branch": "Stable-FCI", **{k: rf_fci_test_metrics[k] for k in comparison_metrics}},
    {"model": "Random Forest", "branch": "All symptoms", **{k: rf_all_test_metrics[k] for k in comparison_metrics}},
    {"model": "XGBoost", "branch": "Stable-FCI", **{k: xgb_fci_test_metrics[k] for k in comparison_metrics}},
    {"model": "XGBoost", "branch": "All symptoms", **{k: xgb_all_test_metrics[k] for k in comparison_metrics}},
])
model_head_to_head.to_csv(RUN_DIR / "model_comparison_summary.csv", index=False)
print(model_head_to_head.to_string(index=False))

model_colors = {"Logistic Regression": "#4C72B0", "Random Forest": "#DD8452", "XGBoost": "#55A868"}
bar_colors = model_head_to_head["model"].map(model_colors)
labels = model_head_to_head["model"] + " (" + model_head_to_head["branch"] + ")"

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(labels, model_head_to_head["f1_macro"], color=bar_colors)
ax.set_ylabel("TEST macro-F1")
ax.set_title("Logistic Regression vs. Random Forest vs. XGBoost — TEST macro-F1")
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.grid(True, axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(RUN_DIR / "model_comparison_barplot.pdf", format="pdf", bbox_inches="tight")
plt.show()


## 21. Notes & Limitations

- **Primary anti-overfitting controls:** a stratified held-out test set, 5-fold stratified cross-validation for `C`, and L2 regularization.
- **Random Forest (Section 22-29):** added as a second downstream model using the same held-out test set, the same 5-fold stratified CV (now tuning `n_estimators` / `max_depth`), and `min_samples_leaf` as its regularizer, so it is directly comparable to Logistic Regression.
- **XGBoost (Section 31-38):** added as a third downstream model, tuned by the same 5-fold stratified CV (over `n_estimators` / `max_depth` / `learning_rate`), with `subsample` and `colsample_bytree` as additional regularizers, so all three models are compared on identical features and splits (Section 39).
- **Stable-FCI uses training data only:** the test set is not used for causal discovery or feature selection.
- **CV nuance:** Stable-FCI feature selection is fixed from the complete training split before Logistic Regression CV. This protects the final test estimate from test leakage, but a fully nested CV would be needed for a strictly unbiased estimate of the entire feature-selection-plus-model-selection pipeline inside the training process.
- **`C` interpretation:** smaller `C` means stronger regularization; larger `C` means weaker regularization.
- **Both branches are tuned separately:** each feature set gets its own best `C`, using the same folds and the same macro-F1 selection criterion.
- **Binary features:** symptoms are already encoded as 0/1, so an additional scaler is not required for this experiment.
- **No-FCI baseline:** all available symptom variables are used; no causal feature selection is applied.
- **No fallback:** if Stable-FCI selects zero disease-adjacent symptoms, the prediction stage stops rather than silently converting the FCI experiment into an all-feature experiment.
- Every run creates a fresh timestamped output directory so previous experiments are preserved.